ដើម្បីរត់សៀវភៅកំណត់ត្រាដែលមានខាងក្រោមនេះ ប្រសិនបើអ្នកមិនទាន់បានធ្វើទេ អ្នកត្រូវតែប្រើម៉ូដែលមួយដែលប្រើ `text-embedding-ada-002` ជាម៉ូដែលមូលដ្ឋាន ហើយកំនត់ឈ្មោះការតម្លើងនៅក្នុងឯកសារ .env ជា `AZURE_OPENAI_EMBEDDINGS_ENDPOINT`


In [ ]:
import os
import pandas as pd
import numpy as np
from openai import AzureOpenAI
from dotenv import load_dotenv

load_dotenv()

client = AzureOpenAI(
  api_key=os.environ['AZURE_OPENAI_API_KEY'],  # this is also the default, it can be omitted
  api_version = "2023-05-15"
  )

model = os.environ['AZURE_OPENAI_EMBEDDINGS_DEPLOYMENT']

SIMILARITIES_RESULTS_THRESHOLD = 0.75
DATASET_NAME = "../embedding_index_3m.json"

បន្ទាប់ យើងងឺកំពុងដំណើរការផ្ទុកចំណត Embedding Index ចូលទៅក្នុង Dataframe របស់ Pandas។ ចំណត Embedding Index ត្រូវបានរក្សាទុកក្នុងឯកសារ JSON មានឈ្មោះ `embedding_index_3m.json`។ ចំណត Embedding Index រួមមាន Embeddings សម្រាប់ឯកសារសន្ទស្សន៍ YouTube ទាំងអស់រហូតដល់ចុងខែ តុលា ឆ្នាំ 2023។


In [ ]:
def load_dataset(source: str) -> pd.core.frame.DataFrame:
    # Load the video session index
    pd_vectors = pd.read_json(source)
    return pd_vectors.drop(columns=["text"], errors="ignore").fillna("")

បន្ទាប់មក យើងនឹងបង្កើតមុខងារ​មួយឈ្មោះ `get_videos` ដែលនឹងស្វែងរកក្នុងអ៊ីនដែគ៏ Embedding សម្រាប់សំណួរ។ មុខងារនេះនឹងបញ្ចេញវីដេអូ ៥ ខ្សែដំបូង ដែលស្រដៀងគ្នាជាមួយសំណួរបំផុត។ មុខងារនោះដំណើរការដូចខាងក្រោម៖

1. ជាដំបូង ធ្វើចម្លងអ៊ីនដែគ៏ Embedding មួយ។
2. បន្ទាប់មក គណនាអ៊ីមបេតធីងសម្រាប់សំណួរប្រើ API អ៊ីមបេតធីង OpenAI។
3. បន្ទាប់មក បង្កើតជួរឈរថ្មីមួយនៅក្នុងអ៊ីនដែគ៏ Embedding ឈ្មោះ `similarity`។ ជួរឈរ `similarity` រួមបញ្ចូលការស្រដៀងគ្នាតាមកូសាញរវាងអ៊ីមបេតធីងសំណួរនិងអ៊ីមបេតធីងសម្រាប់មួយមុខងារវីដេអូ។
4. បន្ទាប់មក ប្រែរើសអ៊ីនដែគ៏ Embedding តាមរយៈជួរឈរ `similarity`។ អ៊ីនដែគ៏ Embedding នឹងត្រូវបានច្រោះឲ្យមានតែវីដេអូដែលមានស្រដៀងគ្នាតាមកូសាញធំស្មើ ឬ ធំជាង 0.75 ប៉ុណ្ណោះ។
5. ចុងក្រោយ អ៊ីនដែគ៏ Embedding នឹងត្រូវបានបង្ហាប់តាមជួរឈរ `similarity` ហើយត្រូវបញ្ចេញវីដេអូ ៥ ខ្សែដំបូង។


In [ ]:
def cosine_similarity(a, b):
    if len(a) > len(b):
        b = np.pad(b, (0, len(a) - len(b)), 'constant')
    elif len(b) > len(a):
        a = np.pad(a, (0, len(b) - len(a)), 'constant')
    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))

def get_videos(
    query: str, dataset: pd.core.frame.DataFrame, rows: int
) -> pd.core.frame.DataFrame:
    # create a copy of the dataset
    video_vectors = dataset.copy()

    # get the embeddings for the query    
    query_embeddings = client.embeddings.create(input=query, model=model).data[0].embedding

    # create a new column with the calculated similarity for each row
    video_vectors["similarity"] = video_vectors["ada_v2"].apply(
        lambda x: cosine_similarity(np.array(query_embeddings), np.array(x))
    )

    # filter the videos by similarity
    mask = video_vectors["similarity"] >= SIMILARITIES_RESULTS_THRESHOLD
    video_vectors = video_vectors[mask].copy()

    # sort the videos by similarity
    video_vectors = video_vectors.sort_values(by="similarity", ascending=False).head(
        rows
    )

    # return the top rows
    return video_vectors.head(rows)

មុខងារនេះដូចជាពិសេសណាស់ វាធ្វើតែបង្ហាញលទ្ធផលនៃការស្វែងរក។


In [ ]:
def display_results(videos: pd.core.frame.DataFrame, query: str):
    def _gen_yt_url(video_id: str, seconds: int) -> str:
        """convert time in format 00:00:00 to seconds"""
        return f"https://youtu.be/{video_id}?t={seconds}"

    print(f"\nVideos similar to '{query}':")
    for _, row in videos.iterrows():
        youtube_url = _gen_yt_url(row["videoId"], row["seconds"])
        print(f" - {row['title']}")
        print(f"   Summary: {' '.join(row['summary'].split()[:15])}...")
        print(f"   YouTube: {youtube_url}")
        print(f"   Similarity: {row['similarity']}")
        print(f"   Speakers: {row['speaker']}")

1. ជំហានដំបូង គេយកតារាងបញ្ចូល (Embedding Index) ឡើងក្នុង Dataframe របស់ Pandas។
2. បន្ទាប់មក អ្នកប្រើត្រូវបានស្នើឲ្យបញ្ចូលសំណួរ។
3. បន្ទាប់មកហៅមុខងារ `get_videos` ដើម្បីស្វែងរកក្នុងតារាងបញ្ចូលសម្រាប់សំណួរ។
4. ចុងក្រោយហៅមុខងារ `display_results` ដើម្បីបង្ហាញលទ្ធផលទៅអ្នកប្រើ។
5. បន្ទាប់មក អ្នកប្រើត្រូវបានស្នើឲ្យបញ្ចូលសំណួរថ្មីមួយទៀត។ ដំណើរការនេះបន្តរហូតដល់អ្នកប្រើបញ្ចូល `exit`។

![](../../../../translated_images/km/notebook-search.1e320b9c7fcbb0bc.webp)

អ្នកត្រូវបានស្នើឲ្យបញ្ចូលសំណួរ។ សូមបញ្ចូលសំណួរមួយ ហើយចុច Enter។ កម្មវិធីនឹងបង្ហាញបញ្ជីវីដេអូនៃវីដេអូដែលពាក់ព័ន្ធនឹងសំណួរនោះ។ កម្មវិធីនឹងផ្តល់តំណរភ្ជាប់ទៅកន្លែងក្នុងវីដេអូនោះដែលមានចម្លើយសំណួរនោះផងដែរ។

នេះគឺជាសំណួរមួយចំនួនដែលអាចសាកល្បងបាន៖

- តើ Azure Machine Learning មានអ្វីខ្លះ?
- តើបណ្តាញប្រសាទកុងវ៉ូលូសិន (convolutional neural networks) ធ្វើការយ៉ាងដូចម្តេច?
- តើបណ្តាញប្រសាទ (neural network) ជាអ្វី?
- តើខ្ញុំអាចប្រើ Jupyter Notebooks ជាមួយ Azure Machine Learning បានទេ?
- តើ ONNX ជាអ្វី?


In [ ]:
pd_vectors = load_dataset(DATASET_NAME)

# get user query from input
while True:
    query = input("Enter a query: ")
    if query == "exit":
        break
    videos = get_videos(query, pd_vectors, 5)
    display_results(videos, query)

---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**ការបដិសេធ**៖  
ឯកសារនេះត្រូវបានបកប្រែដោយប្រើសេវាបកប្រែ AI [Co-op Translator](https://github.com/Azure/co-op-translator)។ ខណៈពេលដែលយើងខិតខំរកភាពត្រឹមត្រូវ សូមយល់ដឹងថា ការបកប្រែដោយស្វ័យប្រវត្តិអាចមានកំហុស ឬភាពមិនត្រឹមត្រូវ។ ឯកសារដើមក្នុងភាសាទទួលស្គាល់គួរត្រូវបានគិតថាជាមូលដ្ឋានសិទ្ធិសម្រាប់ព័ត៌មាន។ សម្រាប់ព័ត៌មានសំខាន់ៗ យើងបានណែនាំឲ្យបកប្រែដោយអ្នកជំនាញមនុស្ស។ យើងមិនទទួលខុសត្រូវចំពោះការយល់ច្រឡំ ឬការបកប្រែខុសពីការប្រើប្រាស់ការបកប្រែនេះឡើយ។
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
